# 04. Epoching

> Detect cue onsets from CH62 (paradigm channel) and segment continuous, preprocessed ECoG into per-trial epochs aligned to gesture cue.

Returns a `(trials × channels × time)` tensor plus class labels (0=relax, 1=fist, 2=peace, 3=open) ready for feature extraction. The 90-trial paradigm gives roughly balanced classes once relax baselines are included or excluded as needed.

In [ ]:
#| default_exp epoching

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import numpy as np

## Cue-onset detection

An onset is a sample where the paradigm channel transitions from `0` (rest) to a non-zero gesture code. The label at the onset sample is the gesture class for that trial.

In [ ]:
#| export
def find_cue_onsets(labels):
    """Return (onsets, classes) — sample indices and gesture codes for every cue in `labels`."""
    labels = np.asarray(labels)
    onsets = np.where((labels[1:] != 0) & (labels[:-1] == 0))[0] + 1
    classes = labels[onsets]
    return onsets, classes

## Epoch extraction

Slice fixed windows around each onset. `tmin` and `tmax` are in seconds relative to the cue (negative `tmin` keeps a pre-cue baseline, useful for normalization). Trials whose window would fall off either edge of the recording are dropped.

In [ ]:
#| export
def epoch(x, onsets, fs, tmin=-0.5, tmax=2.0):
    """Slice (channels, time) array `x` into (n_trials, channels, n_samples) epochs.

    Returns `(epochs, kept_mask)` — `kept_mask` is True for onsets whose window fit."""
    a = int(round(tmin * fs))
    b = int(round(tmax * fs))
    n_samples = b - a
    T = x.shape[-1]
    kept = (onsets + a >= 0) & (onsets + b <= T)
    starts = onsets[kept] + a
    epochs = np.stack([x[..., s:s + n_samples] for s in starts])
    return epochs, kept

## Convenience wrapper

`epoch_recording(rec, tmin, tmax)` does the common case: find cues in `rec.labels`, slice epochs from `rec.ecog`, return `(epochs, classes)` aligned 1-to-1.

In [ ]:
#| export
def epoch_recording(rec, tmin=-0.5, tmax=2.0, signal=None):
    """Epoch `signal` (defaults to `rec.ecog`) using cues from `rec.labels`."""
    x = rec.ecog if signal is None else signal
    onsets, classes = find_cue_onsets(rec.labels)
    epochs, kept = epoch(x, onsets, rec.fs, tmin=tmin, tmax=tmax)
    return epochs, classes[kept]

## Visual validation

Average preprocessed ECoG across trials of each class for a single sensorimotor channel — the classic ERP-style view. Distinct per-class deflections after cue onset are the first sign that the data is decodable.

In [ ]:
%config InlineBackend.figure_format = 'retina'

import matplotlib.pyplot as plt
from br41n_ecog_hand_pose.data import load_ecog, GESTURE_NAMES
from br41n_ecog_hand_pose.preprocessing import preprocess

plt.rcParams.update({
    'axes.grid':      True,
    'grid.linestyle': ':',
    'grid.linewidth': 0.5,
    'grid.alpha':     0.6,
})

In [ ]:
#| eval: false
rec = load_ecog()
clean, _ = preprocess(rec.ecog, rec.fs)
epochs, classes = epoch_recording(rec, tmin=-0.5, tmax=2.0, signal=clean)
print(f'epochs:  {epochs.shape}     # (n_trials, n_channels, n_samples)')
print(f'classes: {classes.shape}    # one label per trial')
for c in [1, 2, 3]:
    print(f'  {GESTURE_NAMES[c]:>5}: {(classes == c).sum()} trials')

In [ ]:
#| eval: false
ch = 30
n_samples = epochs.shape[-1]
t = np.linspace(-0.5, 2.0, n_samples)

fig, ax = plt.subplots(figsize=(8, 4))
for c in [1, 2, 3]:
    mean_trace = epochs[classes == c, ch].mean(axis=0)
    ax.plot(t, mean_trace, label=GESTURE_NAMES[c])
ax.axvline(0, color='k', lw=0.8, ls='--', label='cue onset')
ax.set_xlabel('time from cue onset (s)')
ax.set_ylabel(f'channel {ch} (\u03bcV)')
ax.set_title('Trial-averaged preprocessed ECoG, by class')
ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()